# **Higher Jacobian Matrix**

*This notebook provides the Python implementation used to calculate the Higher Jacobian Matrix.*



First we need to include several important libraries as follows:

In [1]:
from math import factorial
from itertools import product
from sympy import (symbols, diff, SparseMatrix)

The details will no be explain, but you can find the details on [Tenorio](https://AlexTenoVaz.github.io/).

The following source code compute the sets of index of columns and rows, respectively, in a particular order.

The funtion **custom_sort** is requiered to stablish the lexicogrphic graded order. This funtion only need a *list_of_vectors* as a parameter.

The funtions **default_col_vec** and **default_row_vec** compute the indexes mentioned before. As you can see, these functions requiered two parameters, *the_number_of_variables* and *the_order_of_the_jacobian_matrix*, both are necessary, bacuse the size of Higher Jacobian Matrix depends directly on these parameters.

In [2]:
x, y, z, t = symbols('x y z t') # We need define this variables.

def custom_sort(vectors):
    return sorted(vectors, key=lambda v: (sum(v),) + v[::-1])


def default_col_vec( number_of_vars=3,  jac_order=3 ):
    natural_numbers = list(range(jac_order + 1))
    return custom_sort([
        vec for vec in product(natural_numbers, repeat=number_of_vars)
        if 1 <= sum(vec) <= jac_order
    ])

def default_row_vec(
        number_of_vars=3,
        jac_order=4
        ):
    natural_numbers = list(range(jac_order + 1))
    return custom_sort([
        vec for vec in product(natural_numbers, repeat=number_of_vars)
        if 0 <= sum(vec) <= jac_order - 1
    ])

Now, each entrance of th Higher Jacobian Maxtrix is compute using the diffence between the index of its column and the index of its row as follow:

The entrance $(β, α)$ where $α$ is a column index and $β$ is a row index is:

\begin{equation}
			\operatorname{Jac}^{(m)}_{\beta\alpha}(f) :=
			\begin{cases}
				\dfrac{1}{(\alpha - \beta)!} \dfrac{\partial^{\alpha - \beta} f}{\partial x^{\alpha - \beta}} \pmod{f} & \text{if } \alpha \geq \beta, \\[10pt]
				0 & \text{otherwise,}
			\end{cases}+
\end{equation}
where $\alpha \geq \beta$ means $\alpha_i \ge \beta_i$ for all $i=1,\ldots,s$.

The folllowing function computes this, it requiere several parameters:
- The parameter **f** is the polynomial for which you need the Higher Jacobian Matrix.
-  **row_vectors** and **col_vectors** are the sets of indexes for rows and columns, respectly.
- **variables_list** is the set of varibles, the source code is thinking to work with just 3 variables, but you can change this number.
-**number_of_vars** and **jac_order_local** are the number of variables and the jacobian order (local, i.e. just for this function).

The function "derivative_dict" will help us to reduce the runtime, since it avoids performing the same calculation on every entry in the matrix.

As explained in [Tenorio](https://AlexTenoVaz.github.io/) starting with the second row we only have to "reorder" the entries of the first row, so the function just computes up to *jac_order*-th derivatives of a polinomial and assigned a label to it.

In [3]:
def derivative_dictionary(
        f,
        fac = False,
        col_vectors = None,
        variables_list = None,
        number_of_vars=3,
        jac_order_local=4):

    if variables_list is None:
        base_vars = (x, y, z)
        variables_list = list(base_vars[:number_of_vars])

    if col_vectors is None:
        col_vectors = default_col_vec(number_of_vars=number_of_vars, jac_order=jac_order_local)

    if col_vectors is None:
        col_vectors = default_col_vec()
    derivative_dict = {}

    if fac == False:
        for vec in col_vectors:
            deriv = f
            for i, order in enumerate(vec):
                deriv = diff(deriv, variables_list[i], order)
            if deriv != 0:
                derivative_dict[vec] = deriv
    else:
        derivative_dict = {}
        for vec in col_vectors:
            deriv = f
            factorial_vector = 1
            for h in range(len(vec)):
                factorial_vector *= factorial(vec[h])
            for i, order in enumerate(vec):
                deriv = diff(deriv, variables_list[i], order)
            deriv /= factorial_vector
            if deriv != 0:
                derivative_dict[vec] = deriv
    return derivative_dict

def jacobian_matrix(f, row_vectors = None, col_vectors = None,  variables_list=None, number_of_vars=3, jac_order_local=4):

    if row_vectors is None:
        row_vectors = default_row_vec(number_of_vars=number_of_vars, jac_order=jac_order_local)
    if col_vectors is None:
        col_vectors = default_col_vec(number_of_vars=number_of_vars, jac_order=jac_order_local)

    if variables_list is None:
        # por defecto usamos los símbolos globales x,y,z (o menos si number_of_vars<3)
        base_vars = (x, y, z)
        variables_list = list(base_vars[:number_of_vars])

    derivative_dict = derivative_dictionary(f, fac=True, col_vectors=col_vectors, variables_list=variables_list, number_of_vars=number_of_vars, jac_order_local=jac_order_local)

    derivative_dict = derivative_dictionary(f, True)
    num_rows = len(row_vectors)
    num_cols = len(col_vectors)

    sparse_entries = {}

    for r_idx, row in enumerate(row_vectors):
        for c_idx, col in enumerate(col_vectors):
            subtracted_vector = tuple(c - r for r, c in zip(row, col))

            if all(entry >= 0 for entry in subtracted_vector):
                matching_derivative = derivative_dict.get(subtracted_vector, 0)
                if matching_derivative != 0:
                    sparse_entries[(r_idx, c_idx)] = matching_derivative

    M = SparseMatrix(num_rows, num_cols, sparse_entries)
    return M

# Example
Thats all you need to compute the Higher Jacobian Matrix of a polynomial. Now if you want to see an example you need <span style="color:red">compile all previous cells of code</span>, then you execute the following, you will see the Jacobian Matyrix of order 3 of the polynomial $f=x^2+y^3+z^4$.



In [6]:
number_of_vars = 3
jac_order = 3
f = x**2 + y**3 + z**4

row_vec = default_row_vec(number_of_vars, jac_order)
col_vec = default_col_vec(number_of_vars, jac_order)

jac_mat = jacobian_matrix(f, row_vec, col_vec, None, number_of_vars, jac_order)

display(jac_mat)

Matrix([
[2*x, 3*y**2, 4*z**3,   1,      0,    3*y,      0,      0, 6*z**2,   0,      0,      0,      1,      0,      0,      0,      0,      0,    4*z],
[  0,      0,      0, 2*x, 3*y**2,      0, 4*z**3,      0,      0,   1,      0,    3*y,      0,      0,      0,      0, 6*z**2,      0,      0],
[  0,      0,      0,   0,    2*x, 3*y**2,      0, 4*z**3,      0,   0,      1,      0,    3*y,      0,      0,      0,      0, 6*z**2,      0],
[  0,      0,      0,   0,      0,      0,    2*x, 3*y**2, 4*z**3,   0,      0,      0,      0,      1,      0,    3*y,      0,      0, 6*z**2],
[  0,      0,      0,   0,      0,      0,      0,      0,      0, 2*x, 3*y**2,      0,      0, 4*z**3,      0,      0,      0,      0,      0],
[  0,      0,      0,   0,      0,      0,      0,      0,      0,   0,    2*x, 3*y**2,      0,      0, 4*z**3,      0,      0,      0,      0],
[  0,      0,      0,   0,      0,      0,      0,      0,      0,   0,      0,    2*x, 3*y**2,      0,      0, 4*z**3,  